# Local Support Assembly Demo

This notebook demonstrates exact support-closure local assembly. Python handles partitioning, support-patch construction, DoF/index bookkeeping, visualization, and verification. C++ only performs the low-level finite element assembly over a supplied support patch.

In [ ]:
import numpy as np
from ngsolve import *
from ngsolve.webgui import Draw
from netgen.geom2d import unit_square

from partition.metis import metis_partition_from_fes
from utils.patches import build_support_patches, draw_patch, print_patch_summary

import myassembling
print(myassembling)
print(myassembling.__file__)
print(dir(myassembling))

SetNumThreads(1)

## Mesh, Space, And Integrator

`bfi` is a generic NGSolve `BilinearFormIntegrator`. The C++ local assembler does not know the PDE; it only calls `CalcElementMatrix` on this integrator. `SymbolicBFI(grad(u) * grad(v))` defines the Poisson stiffness matrix using NGSolve's built-in symbolic integrator.

In [ ]:
mesh = Mesh(unit_square.GenerateMesh(maxh=0.12, quad_dominated=False))
fes = H1(mesh, order=1, dirichlet="left|right|bottom|top")
u, v = fes.TnT()
bfi = SymbolicBFI(grad(u) * grad(v))

print("ne =", mesh.ne, "ndof =", fes.ndof)

## METIS Element Partition

The helper in `partition/metis.py` builds a shared-DoF element graph and calls METIS. `core_partition` is non-overlapping. `overlapping_partition` is an optional graph-expanded partition for experiments; this expansion is separate from exact support-closure construction.

In [ ]:
core_partition, overlapping_partition, cutcount = metis_partition_from_fes(
    fes,
    nparts=3,
    overlap_width=1,
    free_dofs_only=False,
)

# Use the METIS core partition for the exact support-closure demo.
# For experiments, set partition = overlapping_partition.
partition = overlapping_partition

print("METIS cutcount =", cutcount)
print("core element counts =", [len(p) for p in core_partition])
print("overlapping element counts =", [len(p) for p in overlapping_partition])

## Visualize The Partition

Each plot shows one subdomain. Value `1.0` marks elements in `core_partition[i]`. Value `1.5` would mark elements present only in `overlapping_partition[i]`. With `overlap_width=0`, only the core elements are shown.

In [ ]:
l2 = L2(mesh, order=0)

for i, elements in enumerate(overlapping_partition):
    omega_i = GridFunction(l2, name=f"Omega_{i}")
    omega_i.vec[:] = 0

    core_set = set(core_partition[i])
    overlap_set = set(elements)

    for elnr in overlap_set - core_set:
        omega_i.vec[l2.GetDofNrs(ElementId(VOL, elnr))[0]] = 1.5
    for elnr in core_set:
        omega_i.vec[l2.GetDofNrs(ElementId(VOL, elnr))[0]] = 1.0

    Draw(omega_i, mesh, f"subdomain {i}: core=1, graph-expanded-only=1.5")

## Build Support Patches In Python

For each selected element set, Python constructs the exact support closure:

- `core_dofs`: all global DoFs appearing on `core_elements`
- `support_elements`: all elements whose DoF list intersects `core_dofs`
- `support_dofs`: all global DoFs appearing on `support_elements`
- `core_in_support`: local indices of `core_dofs` inside `support_dofs`

In [ ]:
support_patches = build_support_patches(fes, partition)

for i, patch in enumerate(support_patches):
    print(f"Omega_{i}:")
    print_patch_summary(patch)
    for k, g in enumerate(patch.core_dofs):
        assert patch.support_dofs[patch.core_in_support[k]] == g

## Assemble Local Matrices In C++

The C++ function receives the support patch computed above. It builds only the local sparse matrix by looping over `support_elements`, computing element matrices with `bfi`, and compressing global DoFs through `support_dofs`.

In [ ]:
local_mats = []
for patch in support_patches:
    local = myassembling.MyAssembleGivenLocalSupportMatrix(
        fes,
        bfi,
        patch.core_elements,
        patch.support_elements,
        patch.core_dofs,
        patch.support_dofs,
        patch.core_in_support,
    )
    local_mats.append(local)

for i, local in enumerate(local_mats):
    print(f"Omega_{i}: matrix shape =", (local.mat.height, local.mat.width))

## Visualize Support Patches

Each plot shows one support patch. Value `1` marks input core elements. Value `2` marks support-only elements that are outside the core but contribute to entries involving the core DoFs.

In [ ]:
for i, patch in enumerate(support_patches):
    draw_patch(mesh, patch, name=f"Omega_{i}: core=1, support-only=2")

## Verify Against Global Assembly

The required equality is only the core-core block:

`A_support[core_in_support, core_in_support] == A_global[core_dofs, core_dofs]`

We do not compare `A_support` with `A_global[support_dofs, support_dofs]`; that full support block equality is not required.

In [ ]:
A_global = myassembling.MyAssembleMatrix(fes, bfi)

def dense_submatrix(A, rows, cols):
    return np.array([[A[i, j] for j in cols] for i in rows], dtype=float)

for i, (patch, local) in enumerate(zip(support_patches, local_mats)):
    A_support = np.array(local.mat.ToDense(), dtype=float)
    ids = list(patch.core_in_support)
    A_core_from_support = A_support[np.ix_(ids, ids)]
    A_core_from_global = dense_submatrix(A_global, patch.core_dofs, patch.core_dofs)
    err = np.linalg.norm(A_core_from_support - A_core_from_global, ord=np.inf)
    print(f"Omega_{i}: ||A_support[core,core] - A_global[core,core]||_inf = {err:.3e}")
    assert err < 1e-12